# C7: Logika wtyczki, Processing i projekt koncowy
### Programowanie w GIS — Cwiczenia

---

> **Kurs:** Programowanie w GIS (QGIS)  
> **Cwiczenie:** C7  
> **Powiazany wyklad:** W6  
> **Czas:** ~90 minut  

---

### Cele cwiczenia

Po ukonczeniu tego cwiczenia bedziesz potrafil/a:

- wywolywac algorytmy Processing Toolbox z poziomu wtyczki,
- walidowac dane wejsciowe i obslugiwac bledy,
- zapisywac wyniki do pliku przez dialog wyboru pliku,
- korzystac z `messageBar` do informowania uzytkownika o postepie,
- wybrac i opisac temat projektu koncowego.

---

### Przygotowanie

Kontynuujemy prace na wtyczce z C6.
Upewnij sie ze:

- folder `moja_pierwsza_wtyczka/` istnieje w katalogu wtyczek,
- wtyczka laduje sie poprawnie w QGIS,
- Plugin Reloader jest skonfigurowany na `moja_pierwsza_wtyczka`.


---
<a id='s1'></a>

## 1. Dodanie Processing do wtyczki

### 1.1 Jak dziala processing.run() wewnatrz wtyczki

Modul `processing` jest dostepny wewnatrz wtyczki tak samo jak w konsoli.
Wystarczy go zaimportowac na poczatku pliku:

```python
import processing
```

Kazdy algorytm zwraca slownik — klucz `'OUTPUT'` zawiera wynik.
Jesli podasz `'TEMPORARY_OUTPUT'`, wynik to warstwa w pamieci.

### Zadanie 1.1 — Dodaj buforowanie do wtyczki

Rozbuduj swoja wtyczke z poprzednich cwiczen. Dodaj do dialogu:

- `QDoubleSpinBox` o nazwie `spbPromien` (promien bufora w km),
- etykiete `QLabel` z tekstem `Promien bufora (km):`.

Zaktualizuj `dialog_base.ui` w Qt Designer, zapisz
i przeladuj wtyczke.

Nastepnie dodaj do klasy glownej metode `wykonaj_bufor()`:

```python
def wykonaj_bufor(self):
    import processing
    from qgis.core import (
        QgsCoordinateReferenceSystem, QgsCoordinateTransform, QgsProject
    )

    warstwa = self.iface.activeLayer()
    promien_km = self.window.spbPromien.value()

    bar = self.iface.messageBar()
    bar.pushInfo('Wtyczka', f'Buforowanie ({promien_km} km)...')

    # Reprojekcja do EPSG:3857 (metry)
    r1 = processing.run('native:reprojectlayer', {
        'INPUT'     : warstwa,
        'TARGET_CRS': 'EPSG:3857',
        'OUTPUT'    : 'TEMPORARY_OUTPUT'
    })

    # Bufor w metrach
    r2 = processing.run('native:buffer', {
        'INPUT'    : r1['OUTPUT'],
        'DISTANCE' : promien_km * 1000,
        'SEGMENTS' : 32,
        'DISSOLVE' : False,
        'OUTPUT'   : 'TEMPORARY_OUTPUT'
    })

    # Cofnij do oryginalnego CRS
    r3 = processing.run('native:reprojectlayer', {
        'INPUT'     : r2['OUTPUT'],
        'TARGET_CRS': warstwa.crs(),
        'OUTPUT'    : 'TEMPORARY_OUTPUT'
    })

    wynik = r3['OUTPUT']
    wynik.setName(f'Bufor_{promien_km}km_{warstwa.name()}')
    QgsProject.instance().addMapLayer(wynik)
    self.iface.mapCanvas().refresh()
    bar.pushSuccess('Wtyczka', f'Bufor {promien_km} km gotowy!')
```

Podlacz `btnAnalizuj` do `wykonaj_bufor` zamiast `analizuj`.


### Zadanie 1.2 — Sprawdz czas wykonania

Dodaj do metody `wykonaj_bufor()` pomiar czasu:

```python
import time

t0 = time.time()
# ... kod buforowania ...
t1 = time.time()

bar.pushSuccess(
    'Wtyczka',
    f'Gotowe w {t1-t0:.2f} s. Obiektow: {wynik.featureCount()}'
)
```

Uruchom dla warstwy panstw z promieniem 100 km.
Zanotuj czas wykonania ponizej.


---
<a id='s2'></a>

## 2. Walidacja danych wejsciowych

### 2.1 Dlaczego walidacja jest konieczna

Uzytkownik moze:

- nie wybrac zadnej warstwy,
- wybrac warstwe rastrowa zamiast wektorowej,
- podac promien 0 lub ujemny,
- nie podac sciezki wyjsciowej.

Bez walidacji wtyczka wywoli niezrozumialy blad lub zawiesi sie.
Z walidacja — wyswietli czytelny komunikat i nie wykona operacji.

### Zadanie 2.1 — Napisz metode waliduj()

Dodaj do klasy glownej:

```python
def waliduj(self):
    from qgis.core import QgsMapLayer

    bar   = self.iface.messageBar()
    ok    = True
    bledy = []

    warstwa = self.iface.activeLayer()
    if warstwa is None:
        bledy.append('Nie wybrano zadnej warstwy.')
        ok = False
    elif warstwa.type() != QgsMapLayer.VectorLayer:
        bledy.append('Wybrana warstwa musi byc wektorowa.')
        ok = False

    promien = self.window.spbPromien.value()
    if promien <= 0:
        bledy.append('Promien musi byc wiekszy od zera.')
        ok = False

    if not ok:
        bar.pushWarning('Wtyczka', ' | '.join(bledy))

    return ok
```

Wywolaj `waliduj()` na poczatku `wykonaj_bufor()`:

```python
def wykonaj_bufor(self):
    if not self.waliduj():
        return
    # reszta kodu...
```


**Pytanie 2.1** — Przetestuj walidacje: sprobuj uruchomic wtyczke
bez wybranej warstwy oraz z promieniem 0.
Czy komunikaty sa wystarczajaco zrozumiale dla uzytkownika?
Co bys zmienil/zmienila w tresci komunikatow?


In [ ]:
# Twoja odpowiedz:


### Zadanie 2.2 — Samodzielne: dodatkowe sprawdzenie

Dodaj do metody `waliduj()` sprawdzenie czy warstwa
ma co najmniej jeden obiekt (`featureCount() > 0`).
Buforowanie pustej warstwy jest technicznie poprawne
ale bezuzyteczne — warto o tym poinformowac uzytkownika.

Wyswietl ostrzezenie (`pushWarning`) ale nie blokuj wykonania
— niech uzytkownik zdecyduje czy chce kontynuowac.

Wskazowka: mozesz zwrocic trzy stany: `'ok'`, `'warning'`, `'error'`
zamiast tylko `True`/`False`.


In [ ]:
# Twoja rozbudowana metoda waliduj():

# [QGIS]


---
<a id='s3'></a>

## 3. Zapis wynikow do pliku

### Zadanie 3.1 — Dodaj wybor pliku wyjsciowego

Dodaj do `dialog_base.ui` w Qt Designer:

- `QLineEdit` o nazwie `ledSciezka` (placeholder: `Wybierz plik...`)
- `QPushButton` o nazwie `btnPlik` (tekst: `...`, maxWidth: 40)

Zapisz, przeladuj wtyczke.

### Zadanie 3.2 — Podlacz dialog wyboru pliku

W metodzie `run()` po stworzeniu dialogu dodaj:

```python
self.window.btnPlik.clicked.connect(self.wybierz_plik)
```

Dodaj metode `wybierz_plik()`:

```python
def wybierz_plik(self):
    from qgis.PyQt.QtWidgets import QFileDialog

    sciezka, _ = QFileDialog.getSaveFileName(
        self.window,
        'Zapisz wynik jako',
        '',
        'GeoPackage (*.gpkg);;Shapefile (*.shp)'
    )
    if sciezka:
        if not (sciezka.endswith('.gpkg') or sciezka.endswith('.shp')):
            sciezka += '.gpkg'
        self.window.ledSciezka.setText(sciezka)
```

### Zadanie 3.3 — Dodaj zapis do metody wykonaj_bufor()

Na koncu `wykonaj_bufor()`, po stworzeniu warstwy wynikowej:

```python
sciezka = self.window.ledSciezka.text().strip()

if sciezka:
    from qgis.core import QgsVectorFileWriter, QgsProject
    import os

    driver = 'GPKG' if sciezka.endswith('.gpkg') else 'ESRI Shapefile'
    opcje = QgsVectorFileWriter.SaveVectorOptions()
    opcje.driverName   = driver
    opcje.fileEncoding = 'UTF-8'

    blad, kom, _, _ = QgsVectorFileWriter.writeAsVectorFormatV3(
        wynik, sciezka,
        QgsProject.instance().transformContext(), opcje
    )

    if blad == QgsVectorFileWriter.NoError:
        bar.pushSuccess(
            'Wtyczka', f'Zapisano: {os.path.basename(sciezka)}'
        )
    else:
        bar.pushCritical('Wtyczka', f'Blad zapisu: {kom}')
else:
    bar.pushInfo('Wtyczka', 'Wynik dodany do projektu (brak sciezki zapisu).')
```


---
<a id='s4'></a>

## 4. Komunikaty i obsluga bledow


```python
bar = self.iface.messageBar()

bar.pushInfo('Tytul',     'Informacja')     # niebieski
bar.pushSuccess('Tytul', 'Sukces')          # zielony
bar.pushWarning('Tytul', 'Ostrzezenie')     # zolty
bar.pushCritical('Tytul','Blad krytyczny')  # czerwony
```

---
<a id='s6'></a>

## 6. Projekt koncowy — tematy i wymagania

Projekt koncowy to samodzielnie napisana wtyczka QGIS.
Wybierz jeden temat z ponizszej listy lub zaproponuj wlasny.

---

### Wymagania wspolne dla wszystkich projektow

Kazda wtyczka musi zawierac:

| Element | Opis |
|---|---|
| Struktura plikow | Poprawne `__init__.py`, `metadata.txt`, klasa glowna |
| Interfejs GUI | Co najmniej 4 widgety wejsciowe, layout, czytelne etykiety |
| QgsMapLayerComboBox | Wybor warstwy wejsciowej z projektu |
| Walidacja | Sprawdzenie danych przed wykonaniem, komunikaty o bledach |
| Logika operacji | Przynajmniej jeden algorytm Processing lub operacja PyQGIS |
| Zapis wyniku | Nowa warstwa w projekcie lub plik na dysku |
| messageBar | Komunikaty info/success/warning/critical w odpowiednich miejscach |
| metadata.txt | Wypelnione wszystkie wymagane pola + opis `about` |
| Komentarze w kodzie | Kluczowe fragmenty opisane po polsku lub angielsku |

---

### Tematy do wyboru

#### Temat 1 — Analizator warstwy
*Poziom: podstawowy*

Wtyczka wyswietla szczegolowy raport o wybranej warstwie wektorowej:
typ geometrii, liczba obiektow, pola atrybutow z typami,
zasieg przestrzenny, CRS, pole powierzchni i obwod
(dla poligonow), srednie wartosci pol numerycznych.
Raport zapisywany jest do pliku tekstowego.

**Wymagane algorytmy/operacje:** `QgsDistanceArea`, `bandStatistics` lub petla
po obiektach, `QgsVectorFileWriter` lub zapis przez Python `open()`.

---

#### Temat 2 — Narzedzie buforu z opcjami
*Poziom: podstawowy-sredni*

Rozbudowana wersja buforu z opcjami:
wybor jednostki (m / km), opcja scalania buforow (dissolve),
wybor pola po ktorym scalamy, podglad liczby obiektow wynikowych.

**Wymagane algorytmy:** `native:buffer`, `native:dissolve`,
`native:reprojectlayer`.

---

#### Temat 3 — Kalkulator gestosci punktow
*Poziom: sredni*

Dla warstwy punktowej i warstwy poligonowej oblicza
ile punktow lezy wewnatrz kazdego poligonu (gestosc).
Wynik zapisuje jako nowe pole atrybutu w warstwie poligonowej.
Wizualizuje choropleta wg gestosci.

**Wymagane algorytmy:** `native:countpointsinpolygon`,
`QgsVectorLayer` tryb edycji, `changeAttributeValue`.

---

#### Temat 4 — Generowanie losowych geometrii na mapie
*Poziom: sredni*

Narzędzie do generowania losowych geometrii gdzie użytkownik może podać zakres współrzędnych w których można generować, liczbę obiektów do wygenerowania oraz typ geometrii do generowania.

---

#### Temat 5 — Znajdz sasiadow
*Poziom: sredni*

Uzytkownik klika obiekt na mapie (lub wybiera z listy),
wtyczka znajduje wszystkich sasiadow obiektu w podanym zakresie tolerancji
i zaznacza ich na mapie.

---

#### Temat 6 — Kalkulator stref buforowych
*Poziom: sredni*

Tworzy N koncentrycznych stref buforowych wokol wybranych obiektow.
Uzytkownik podaje liczbe stref i promien kazdej.
Kazda strefa to pierscien (roznica miedzy kolejnymi buforami).
Strefy sa zapisywane jako osobne warstwy lub jako jedna warstwa
z polem `strefa_nr`.

**Wymagane algorytmy:** `native:buffer`, `QgsGeometry.difference()`,
`native:reprojectlayer`, `QgsCategorizedSymbolRenderer`.

---

#### Temat 7 — Mapa dostepnosci
*Poziom: zaawansowany*

Uzytkownik wybiera warstwe punktowa (np. szpitale, szkoly)
i warstwe obszarow (np. gminy) - z otwartych danych.
Wtyczka oblicza dla kazdego obszaru odleglosc do najblizszego punktu
i liczbe punktow w promieniu N km.
Wynik jako choropleta: obszary kolorowane wg dostepnosci.

**Wymagane operacje:** `QgsDistanceArea`, petla po obiektach,
`native:countpointsinpolygon`, `native:buffer`,
`QgsGraduatedSymbolRenderer`.


### Temat własny
Do zaakceptowania przez prowadzącego

### Zatwierdzenie tematu

Do końca tygodnia 29.05.2026 mail na kamil.smolak@upwr.edu.pl z podaniem:
    
    Osób w grupie
    Tematu 
    
### Forma oddania

Spakuj folder wtyczki jako `wtyczka_[numer_indeksu].zip`.
Nazwa folderu wewnatrz ZIP musi byc w snake_case (bez spacji).
Wtyczka musi dac sie zainstalowac przez rozpakowanie do folderu wtyczek
i wlaczenie w Menedzerze — przetestuj to przed oddaniem.


**Termin oddania**: 19.06.2026